# 🤖 CIFAKE: Real vs AI-Generated Image Detector
**Dataset:** [CIFAKE](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images) — 60,000 real images (CIFAR-10) + 60,000 AI-generated synthetic images (Stable Diffusion)

**Task:** Binary classification — `REAL` vs `FAKE`

**Models:**
- CNN with residual blocks
- Vision Transformer (ViT) from scratch

---
**Setup:** Download the dataset from Kaggle before running.
```bash
pip install kaggle
kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
unzip cifake-real-and-ai-generated-synthetic-images.zip -d cifake
```
Expected folder structure:
```
cifake/
  train/
    REAL/   (30,000 images)
    FAKE/   (30,000 images)
  test/
    REAL/   (10,000 images)
    FAKE/   (10,000 images)
```

## 1. Install & Import Dependencies

In [ ]:
!pip install torch torchvision einops matplotlib seaborn scikit-learn kaggle -q

In [ ]:
# ── Optional: download via Kaggle API ──────────────────────────────────────────
# Requires ~/.kaggle/kaggle.json with your API credentials
# import os
# os.system('kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images')
# os.system('unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d cifake')
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)
from einops import rearrange

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
torch.manual_seed(42)

## 2. Dataset Loading

In [ ]:
DATA_ROOT  = Path('./cifake')   # adjust if your path differs
BATCH_SIZE = 128
IMG_SIZE   = 32
CLASS_NAMES = ['FAKE', 'REAL']  # alphabetical = torchvision default

# CIFAKE images are 32x32 — same resolution as CIFAR-10
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std =[0.5, 0.5, 0.5]),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std =[0.5, 0.5, 0.5]),
])

train_dataset = torchvision.datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
test_dataset  = torchvision.datasets.ImageFolder(DATA_ROOT / 'test',  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# class indices assigned by ImageFolder (alphabetical)
print('Class map:', train_dataset.class_to_idx)  # {'FAKE': 0, 'REAL': 1}
print(f'Train: {len(train_dataset):,} | Test: {len(test_dataset):,}')

### Visualise Real vs Fake Samples

In [ ]:
def denorm(tensor):
    return np.clip((tensor.permute(1,2,0).numpy() * 0.5) + 0.5, 0, 1)

# Grab examples of each class
fake_idx = [i for i, (_, l) in enumerate(train_dataset.samples) if l == 0][:8]
real_idx = [i for i, (_, l) in enumerate(train_dataset.samples) if l == 1][:8]

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for col, idx in enumerate(fake_idx):
    img, _ = test_dataset[idx]  # use test transform (no augmentation)
    axes[0, col].imshow(denorm(img))
    axes[0, col].axis('off')
    if col == 0: axes[0, col].set_ylabel('AI FAKE', fontsize=11, color='red', rotation=90, labelpad=10)

for col, idx in enumerate(real_idx):
    img, _ = test_dataset[idx]
    axes[1, col].imshow(denorm(img))
    axes[1, col].axis('off')
    if col == 0: axes[1, col].set_ylabel('REAL', fontsize=11, color='green', rotation=90, labelpad=10)

plt.suptitle('CIFAKE Dataset: AI-Generated (top) vs Real (bottom)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Model 1 — CNN (ResNet-style)

In [ ]:
class ResBlock(nn.Module):
    """Basic residual block with optional downsampling."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_c)
        ) if (in_c != out_c or stride != 1) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.main(x) + self.skip(x))


class FakeDetectorCNN(nn.Module):
    """
    CNN binary classifier for REAL vs FAKE detection.
    Input: 32x32 RGB images
    Output: logit for class [FAKE=0, REAL=1]
    """
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )                                                   # [B, 64, 32, 32]
        self.layer1 = ResBlock(64,  64)                     # [B, 64,  32, 32]
        self.layer2 = ResBlock(64,  128, stride=2)          # [B, 128, 16, 16]
        self.layer3 = ResBlock(128, 256, stride=2)          # [B, 256,  8,  8]
        self.layer4 = ResBlock(256, 512, stride=2)          # [B, 512,  4,  4]
        self.pool   = nn.AdaptiveAvgPool2d(1)               # [B, 512,  1,  1]
        self.head   = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 2),                              # FAKE / REAL
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)


cnn_model = FakeDetectorCNN().to(DEVICE)
cnn_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'CNN parameters: {cnn_params:,}')

## 4. Model 2 — Vision Transformer (ViT)

In [ ]:
class PatchEmbed(nn.Module):
    """Splits 32x32 image into (32/patch_size)^2 patches and projects to embed_dim."""
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                            # [B, E, h, w]
        return rearrange(x, 'b e h w -> b (h w) e') # [B, N, E]


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads,
                                            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden     = int(embed_dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        n = self.norm1(x)
        x = x + self.attn(n, n, n)[0]
        x = x + self.mlp(self.norm2(x))
        return x


class FakeDetectorViT(nn.Module):
    """
    ViT binary classifier for REAL vs FAKE detection.
    patch_size=4  -> 64 patches from a 32x32 image
    """
    def __init__(self, img_size=32, patch_size=4, embed_dim=192,
                 depth=8, num_heads=8, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2),   # FAKE / REAL
        )

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B  = x.shape[0]
        x  = self.patch_embed(x)                          # [B, N, E]
        cls = self.cls_token.expand(B, -1, -1)            # [B, 1, E]
        x  = torch.cat([cls, x], dim=1)                   # [B, N+1, E]
        x  = self.pos_drop(x + self.pos_embed)
        x  = self.blocks(x)
        x  = self.norm(x[:, 0])                           # CLS token only
        return self.head(x)


vit_model = FakeDetectorViT().to(DEVICE)
vit_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'ViT parameters: {vit_params:,}')

## 5. Training Utilities

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                out  = model(images)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_preds, all_labels = [], [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        out   = model(images)
        loss  = criterion(out, labels)
        probs = torch.softmax(out, dim=1)[:, 1]   # P(REAL)
        preds = out.argmax(1)
        total_loss += loss.item() * images.size(0)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_probs), np.array(all_preds), np.array(all_labels)


def train_model(model, train_loader, test_loader,
                epochs=30, lr=3e-4, weight_decay=1e-4, name='Model'):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    history  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, scaler)
        va_loss, va_acc, _, _, _ = evaluate(model, test_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), f'{name}_best.pt')

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{name}] Ep {epoch:3d}/{epochs} | '
                  f'Train: loss={tr_loss:.4f} acc={tr_acc:.4f} | '
                  f'Val: loss={va_loss:.4f} acc={va_acc:.4f}')

    print(f'\n✅ [{name}] Best val accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)')
    return history

print('Utilities ready.')

## 6. Train CNN

In [ ]:
EPOCHS = 30  # Increase to 50 for stronger results

print('='*65)
print('Training CNN — CIFAKE Real vs Fake Detector')
print('='*65)
cnn_history = train_model(
    cnn_model, train_loader, test_loader,
    epochs=EPOCHS, lr=1e-3, name='CNN'
)

## 7. Train ViT

In [ ]:
print('='*65)
print('Training ViT — CIFAKE Real vs Fake Detector')
print('='*65)
vit_history = train_model(
    vit_model, train_loader, test_loader,
    epochs=EPOCHS, lr=3e-4, name='ViT'
)

## 8. Training Curves

In [ ]:
def plot_training_curves(cnn_h, vit_h, epochs):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, epochs + 1)
    colors = {'cnn': '#2196F3', 'vit': '#F44336'}

    ax = axes[0]
    ax.plot(ep, cnn_h['train_loss'], color=colors['cnn'], lw=2,   label='CNN train')
    ax.plot(ep, cnn_h['val_loss'],   color=colors['cnn'], lw=2, ls='--', label='CNN val')
    ax.plot(ep, vit_h['train_loss'], color=colors['vit'], lw=2,   label='ViT train')
    ax.plot(ep, vit_h['val_loss'],   color=colors['vit'], lw=2, ls='--', label='ViT val')
    ax.set(title='Loss (Binary CE)', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(ep, [a*100 for a in cnn_h['train_acc']], color=colors['cnn'], lw=2,   label='CNN train')
    ax.plot(ep, [a*100 for a in cnn_h['val_acc']],   color=colors['cnn'], lw=2, ls='--', label='CNN val')
    ax.plot(ep, [a*100 for a in vit_h['train_acc']], color=colors['vit'], lw=2,   label='ViT train')
    ax.plot(ep, [a*100 for a in vit_h['val_acc']],   color=colors['vit'], lw=2, ls='--', label='ViT val')
    ax.axhline(y=50, color='gray', ls=':', lw=1, label='Chance')
    ax.set(title='Accuracy (%)', xlabel='Epoch', ylabel='Accuracy (%)', ylim=[40, 102])
    ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle('CIFAKE Detector: CNN vs ViT Training Curves', fontsize=14)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_curves(cnn_history, vit_history, EPOCHS)

## 9. Final Evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()

# Load best checkpoints
cnn_model.load_state_dict(torch.load('CNN_best.pt', map_location=DEVICE))
vit_model.load_state_dict(torch.load('ViT_best.pt', map_location=DEVICE))

_, cnn_acc, cnn_probs, cnn_preds, cnn_labels = evaluate(cnn_model, test_loader, criterion)
_, vit_acc, vit_probs, vit_preds, vit_labels = evaluate(vit_model, test_loader, criterion)

print(f'CNN  Test Accuracy: {cnn_acc*100:.2f}%')
print(f'ViT  Test Accuracy: {vit_acc*100:.2f}%')

### 9.1 Confusion Matrices

In [ ]:
def plot_confusion(preds, labels, title, ax):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, cbar=False, linewidths=0.5)
    # Also annotate raw counts
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.72, f'n={cm[i,j]:,}',
                    ha='center', va='center', fontsize=8, color='gray')
    ax.set(title=title, xlabel='Predicted', ylabel='True')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion(cnn_preds, cnn_labels, f'CNN  (acc={cnn_acc*100:.2f}%)', ax1)
plot_confusion(vit_preds, vit_labels, f'ViT  (acc={vit_acc*100:.2f}%)', ax2)
plt.suptitle('CIFAKE — Confusion Matrices (Normalised)', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 ROC Curves & AUC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (probs, labels, name, color) in zip(axes, [
    (cnn_probs, cnn_labels, 'CNN', '#2196F3'),
    (vit_probs, vit_labels, 'ViT', '#F44336'),
]):
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc     = auc(fpr, tpr)
    pr, rc, _   = precision_recall_curve(labels, probs)
    pr_auc      = auc(rc, pr)

    ax.plot(fpr, tpr, color=color, lw=2, label=f'ROC AUC = {roc_auc:.4f}')
    ax.plot([0,1], [0,1], 'k--', lw=1)
    ax.fill_between(fpr, tpr, alpha=0.1, color=color)
    ax.set(title=f'{name} — ROC Curve', xlabel='False Positive Rate',
           ylabel='True Positive Rate', xlim=[-0.01,1.01], ylim=[-0.01,1.05])
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

plt.suptitle('ROC Curves: Real vs AI-Generated Image Detection', fontsize=13)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Classification Reports

In [ ]:
for name, preds, labels in [('CNN', cnn_preds, cnn_labels), ('ViT', vit_preds, vit_labels)]:
    print(f'\n{name} Classification Report')
    print('─'*50)
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

## 10. Qualitative Inspection — What Does Each Model Get Wrong?

In [ ]:
@torch.no_grad()
def get_errors(model, loader, max_errors=32):
    """Return images that the model misclassified."""
    model.eval()
    errors = {'images': [], 'true': [], 'pred': [], 'conf': []}
    for images, labels in loader:
        images_d, labels_d = images.to(DEVICE), labels.to(DEVICE)
        probs = torch.softmax(model(images_d), dim=1)
        preds = probs.argmax(1)
        mask  = preds != labels_d
        for img, true, pred, prob in zip(
                images[mask.cpu()],
                labels[mask.cpu()],
                preds[mask].cpu(),
                probs[mask].cpu()):
            errors['images'].append(img)
            errors['true'].append(true.item())
            errors['pred'].append(pred.item())
            errors['conf'].append(prob[pred.item()].item())
            if len(errors['images']) >= max_errors:
                return errors
    return errors


def plot_errors(errors, model_name, n=16):
    fig, axes = plt.subplots(2, 8, figsize=(18, 5))
    for i, ax in enumerate(axes.flat):
        if i >= min(n, len(errors['images'])): ax.axis('off'); continue
        ax.imshow(denorm(errors['images'][i]))
        ax.set_title(
            f'True: {CLASS_NAMES[errors["true"][i]]}\n'
            f'Pred: {CLASS_NAMES[errors["pred"][i]]} ({errors["conf"][i]:.2f})',
            fontsize=7, color='red'
        )
        ax.axis('off')
    plt.suptitle(f'{model_name} — Misclassified Samples', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{model_name}_errors.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Finding CNN misclassifications...')
cnn_errors = get_errors(cnn_model, test_loader)
plot_errors(cnn_errors, 'CNN')

print('Finding ViT misclassifications...')
vit_errors = get_errors(vit_model, test_loader)
plot_errors(vit_errors, 'ViT')

## 11. Inference on a Single Custom Image

In [ ]:
from PIL import Image

infer_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

@torch.no_grad()
def predict_image(image_path, cnn, vit):
    img = Image.open(image_path).convert('RGB')
    x   = infer_transform(img).unsqueeze(0).to(DEVICE)

    for model, name in [(cnn, 'CNN'), (vit, 'ViT')]:
        model.eval()
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        pred  = CLASS_NAMES[probs.argmax()]
        print(f'[{name}]  FAKE: {probs[0]*100:.1f}%  |  REAL: {probs[1]*100:.1f}%  →  {pred}')

    plt.figure(figsize=(3,3))
    plt.imshow(img)
    plt.title('Input Image', fontsize=11)
    plt.axis('off')
    plt.show()

# ── Usage: replace with your image path ───────────────────────────────────────
# predict_image('path/to/your/image.jpg', cnn_model, vit_model)
# ──────────────────────────────────────────────────────────────────────────────
print('predict_image() is ready. Provide an image path to run inference.')

## 12. Model Comparison Summary

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

cnn_f1  = f1_score(cnn_labels, cnn_preds, average='macro')
vit_f1  = f1_score(vit_labels, vit_preds, average='macro')
cnn_auc = roc_auc_score(cnn_labels, cnn_probs)
vit_auc = roc_auc_score(vit_labels, vit_probs)

print('\n' + '='*60)
print(f'{"Metric":<20} {"CNN":>15} {"ViT":>15}')
print('-'*60)
print(f'{"Parameters":<20} {cnn_params/1e6:>14.2f}M {vit_params/1e6:>14.2f}M')
print(f'{"Test Accuracy":<20} {cnn_acc*100:>14.2f}% {vit_acc*100:>14.2f}%')
print(f'{"Macro F1":<20} {cnn_f1:>15.4f} {vit_f1:>15.4f}')
print(f'{"ROC AUC":<20} {cnn_auc:>15.4f} {vit_auc:>15.4f}')
print('='*60)

# Bar chart
metrics = ['Accuracy (%)', 'Macro F1', 'ROC AUC']
cnn_vals = [cnn_acc*100, cnn_f1*100, cnn_auc*100]
vit_vals = [vit_acc*100, vit_f1*100, vit_auc*100]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, cnn_vals, w, label='CNN', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + w/2, vit_vals, w, label='ViT', color='#F44336', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set(title='CIFAKE Detector — CNN vs ViT', ylim=[0, 110])
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                           f'{bar.get_height():.1f}', ha='center', fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                           f'{bar.get_height():.1f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Notes & Next Steps

### Expected performance (30 epochs, from scratch)
| Model | ~Test Acc |
|---|---|
| CNN (ResNet-style) | ~92–95% |
| ViT (from scratch) | ~87–92% |

### To push higher:
- **More epochs** — set `EPOCHS = 60–100`
- **Stronger augmentation** — Mixup, CutMix, RandAugment
- **Pretrained ViT** via `timm`:
  ```python
  import timm
  model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=2)
  # resize images to 224x224 in transforms
  ```
- **Pretrained EfficientNet/ResNet50** for the CNN arm
- **Ensemble** — average CNN + ViT softmax scores at inference time

### Why CIFAKE is challenging:
Stable Diffusion images can be visually indistinguishable to humans. Models learn subtle artefacts in frequency/texture space — which is why CNNs (strong local texture bias) often outperform ViTs on this task without pretraining.